# Project Sanjay MK2 — Day 2: Resume `police_full_v1` Training

**Resume YOLO11s from epoch 1 → 100** using the 31,538-image merged dataset.

| Item | Detail |
|------|--------|
| Checkpoint | `police_full_v1_epoch1.pt` (55 MB, YOLO11s) |
| Dataset | `data/visdrone_police/` — 21,693 train / 8,235 val |
| Classes | person, weapon_person, vehicle, fire, explosive_device, crowd |
| Target | mAP50 > 0.55 (baseline Day 1: 0.480 on YOLO11n) |

**Pre-requisite (do before running):**
Upload `runs/detect/runs/detect/police_full_v1/weights/last.pt` from your local machine to:
```
My Drive/SanjayMK2/checkpoints/police_full_v1_epoch1.pt
```

## CELL 1 — Install + clone repo

In [ ]:
!pip install ultralytics kagglehub scipy pillow -q
!git clone https://github.com/YOUR_ORG/Sanjay_MK2 /content/Sanjay_MK2
%cd /content/Sanjay_MK2

# Patch YAML with absolute path immediately after clone.
# Ultralytics resolves relative 'path:' values from its own datasets dir
# (/content/datasets), not the project root — this causes the images not found error.
import os
os.makedirs('config/training', exist_ok=True)
with open('config/training/visdrone_police.yaml', 'w') as f:
    f.write(
        "path: /content/Sanjay_MK2/data/visdrone_police\n"
        "train: images/train\n"
        "val: images/val\n"
        "test: images/test\n"
        "names:\n"
        "  0: person\n"
        "  1: weapon_person\n"
        "  2: vehicle\n"
        "  3: fire\n"
        "  4: explosive_device\n"
        "  5: crowd\n"
    )
print("YAML written with absolute path")

# Verify GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## CELL 2 — VisDrone download + remap (~2 GB, ~12 min)

In [ ]:
!python scripts/train_yolo.py --setup-visdrone

## CELL 3 — D-Fire from Kaggle (~2.84 GB, ~5 min)

In [ ]:
import kagglehub, shutil
from pathlib import Path

path = kagglehub.dataset_download('sayedgamal99/smoke-fire-detection-yolo')
FIRE_SRC = Path(path) / 'data'
FIRE_DST = Path('data/supplementary/fire_dfire')

for ss, sd in [('train','train'), ('val','val'), ('test','val')]:
    sl, si = FIRE_SRC/ss/'labels', FIRE_SRC/ss/'images'
    dl, di = FIRE_DST/'labels'/sd, FIRE_DST/'images'/sd
    dl.mkdir(parents=True, exist_ok=True)
    di.mkdir(parents=True, exist_ok=True)
    if not sl.exists(): continue
    for lp in sorted(sl.glob('*.txt')):
        lines = []
        with open(lp) as f:
            for line in f:
                p = line.strip().split()
                if len(p) >= 5:
                    p[0] = '3'  # smoke=0 and fire=1 both → class 3 (fire)
                    lines.append(' '.join(p))
        with open(dl/lp.name, 'w') as f:
            f.write('\n'.join(lines) + '\n' if lines else '')
        stem = lp.stem
        for ext in ['.jpg','.jpeg','.png']:
            isrc = si/(stem+ext)
            if isrc.exists():
                idst = di/(stem+ext)
                if not idst.exists(): shutil.copy2(isrc, idst)
                break
print('D-Fire done — train:', len(list((FIRE_DST/'images'/'train').glob('*'))),
      'val:', len(list((FIRE_DST/'images'/'val').glob('*'))))

## CELL 4 — ShanghaiTech crowd from Kaggle (~333 MB, ~2 min)

In [ ]:
import kagglehub, shutil, numpy as np
from pathlib import Path
from scipy.io import loadmat
from PIL import Image

path = kagglehub.dataset_download('tthien/shanghaitech')
CS = Path(path) / 'ShanghaiTech' / 'part_A'
CD = Path('data/supplementary/crowd_shanghaitech')

for ss, sd in [('train_data','train'), ('test_data','val')]:
    ig = CS/ss/'images'; gt = CS/ss/'ground-truth'
    di = CD/'images'/sd; dl = CD/'labels'/sd
    di.mkdir(parents=True, exist_ok=True); dl.mkdir(parents=True, exist_ok=True)
    if not ig.exists(): continue
    c = 0
    for ip in sorted(ig.glob('*.jpg')):
        gp = gt/f'GT_{ip.stem}.mat'
        if not gp.exists(): continue
        try:
            mat = loadmat(str(gp))
            pts = mat['image_info'][0][0][0][0][0]
            if len(pts) < 5: continue
            im = Image.open(ip); w, h = im.size
            xn = max(0, pts[:,0].min()-20)/w; yn = max(0, pts[:,1].min()-20)/h
            xx = min(w, pts[:,0].max()+20)/w; yx = min(h, pts[:,1].max()+20)/h
            cx=(xn+xx)/2; cy=(yn+yx)/2; bw=xx-xn; bh=yx-yn
            shutil.copy2(ip, di/ip.name)
            with open(dl/f'{ip.stem}.txt','w') as f:
                f.write(f'5 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n')
            c += 1
        except: continue
    print(f'ShanghaiTech {ss}->{sd}: {c} images')

## CELL 5 — Synthetic weapon_person from VisDrone persons

In [ ]:
import shutil, random
from pathlib import Path
random.seed(42)
VS = Path('data/visdrone_police')
WD = Path('data/supplementary/weapon_synthetic')

for split, max_n in [('train',800), ('val',100)]:
    sl = VS/'labels'/split; si = VS/'images'/split
    dl = WD/'labels'/split; di = WD/'images'/split
    dl.mkdir(parents=True, exist_ok=True); di.mkdir(parents=True, exist_ok=True)
    cands = []
    for lp in sorted(sl.glob('*.txt')):
        boxes = [line.strip().split() for line in open(lp) if len(line.strip().split())>=5]
        persons = [i for i,b in enumerate(boxes) if b[0]=='0']
        if persons: cands.append((lp, boxes, persons))
    random.shuffle(cands)
    c = 0
    for lp, boxes, persons in cands[:max_n]:
        stem = lp.stem
        n_wpn = min(random.randint(1,2), len(persons))
        wpn_idx = set(random.sample(persons, n_wpn))
        new_lines = [' '.join(['1']+b[1:]) if i in wpn_idx else ' '.join(b)
                     for i,b in enumerate(boxes)]
        img = next((si/(stem+ext) for ext in ['.jpg','.jpeg','.png']
                    if (si/(stem+ext)).exists()), None)
        if not img: continue
        shutil.copy2(img, di/f'wpn_{stem}{img.suffix}')
        with open(dl/f'wpn_{stem}.txt','w') as f: f.write('\n'.join(new_lines)+'\n')
        c += 1
    print(f'weapon_person {split}: {c}')

## CELL 6 — Merge all supplementary into visdrone_police

Expected result: ~31,538 total images, 5 classes populated.

In [ ]:
!python scripts/prepare_supplementary_data.py --merge-all
!python scripts/train_yolo.py --merge data/supplementary_merged --auto-prefix
!python scripts/audit_dataset.py data/visdrone_police

## CELL 7 — Mount Drive and copy checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copy(
    '/content/drive/MyDrive/SanjayMK2/checkpoints/police_full_v1_epoch1.pt',
    'police_full_v1_epoch1.pt'
)
import os
print('Checkpoint size:', os.path.getsize('police_full_v1_epoch1.pt') / 1e6, 'MB')

## CELL 8 — Resume training (epochs 2–100)

Results save directly to Google Drive — safe against Colab disconnects.

> **If `resume=True` fails** (run config missing after runtime reset): remove `resume=True`, change `epochs=99`.

In [ ]:
from ultralytics import YOLO

model = YOLO('police_full_v1_epoch1.pt')
results = model.train(
    data='config/training/visdrone_police.yaml',
    epochs=100,
    imgsz=640,
    device=0,
    batch=-1,           # auto batch
    patience=20,
    name='police_full_v1',
    project='/content/drive/MyDrive/SanjayMK2/runs',
    resume=True,
    exist_ok=True,
)
print('Best mAP50:', results.results_dict.get('metrics/mAP50(B)'))

## CELL 9 — Download best weights

After training completes, download `best.pt` and copy to:
```
runs/detect/runs/detect/police_full_v1/weights/best_day2.pt
```
Then validate on WSL2:
```bash
python scripts/validate_model.py --yolo best_day2.pt --all --compare 2>&1 | tee reports/day2/validation.log
```

In [ ]:
from google.colab import files
best_path = '/content/drive/MyDrive/SanjayMK2/runs/police_full_v1/weights/best.pt'
files.download(best_path)
print('Downloaded best.pt')
print('Next: copy to runs/detect/runs/detect/police_full_v1/weights/best_day2.pt')
print('Then: python scripts/validate_model.py --yolo best_day2.pt --all --compare')

## Day 2 Success Bar

| Metric | Baseline (Day 1 YOLO11n) | Target (Day 2 YOLO11s) |
|--------|--------------------------|------------------------|
| mAP50 (all) | 0.480 | **> 0.55** |
| mAP50-95 (all) | 0.247 | **> 0.28** |
| person mAP50 | 0.329 | > 0.35 |
| vehicle mAP50 | 0.631 | > 0.65 |
| fire mAP50 | 0.000 | **> 0.40** |
| weapon_person mAP50 | 0.000 | > 0.10 |
| crowd mAP50 | 0.000 | > 0.15 |
| explosive_device mAP50 | 0.000 | 0.000 (deferred) |

## Known Issues

- **Out-of-bounds D-Fire labels:** ~20 images have coords slightly > 1.0. YOLO skips them — non-fatal.
- **weapon_person is synthetic:** Expect low precision until real weapon data is sourced (Roboflow API key required).
- **crowd is ground-level:** ShanghaiTech is not aerial. Performance improves when DroneCrowd is added.
- **explosive_device:** Zero instances — deferred until a real dataset decision is made.
- **`resume=True` requires run `.yaml`** alongside checkpoint. If runtime resets between cells 7 and 8, re-run cell 7. If yaml is missing, set `resume=False` and `epochs=99`.